# Tutorial 5: Evaluation and Analysis（評価と解析）

**所要時間**: 40-50分

**学習内容**:
- 安定性メトリクスの詳細計算
- RDKit統合と検証
- 一意性・新規性アルゴリズム
- 結合長・結合角分布解析
- 包括的評価パイプライン
- 可視化手法

**重要**: フォールバック処理なし。全ての評価は厳密に実行されます。

In [ ]:
# セットアップ
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from configs.datasets_config import get_dataset_info
from qm9.analyze import check_stability, analyze_stability_for_molecules
from qm9 import bond_analyze

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('\n=== Tutorial 5: 評価と解析 ===')
print('フォールバック処理なし - 全て厳密評価')

## 1. 生成分子データの準備

実際の評価では、モデルから生成された分子を使用します。
ここでは、テストデータを作成します。

In [ ]:
# データセット情報の取得
dataset_info = get_dataset_info('qm9', remove_h=False)
print(f'Dataset: {dataset_info["name"]}')
print(f'Atom decoder: {dataset_info["atom_decoder"]}')
print(f'Number of atom types: {len(dataset_info["atom_decoder"])}')

# テスト用の分子データを生成（実際には main_qm9.py からサンプリング）
def create_test_molecule(n_atoms=10, atom_types=None):
    """テスト用の分子を作成（実際の生成をシミュレート）"""
    if atom_types is None:
        # ランダムに原子タイプを選択（H, C, N, O, Fのみ）
        atom_types = np.random.choice([0, 1, 2, 3, 4], size=n_atoms)
    
    # ランダムな3D座標（実際にはモデルが生成）
    positions = np.random.randn(n_atoms, 3) * 2.0
    
    return {
        'positions': positions,
        'atom_types': atom_types
    }

# テスト用分子の生成
test_molecules = [create_test_molecule(n_atoms=np.random.randint(5, 15)) for _ in range(10)]

print(f'\n生成されたテスト分子数: {len(test_molecules)}')
print(f'サンプル分子の原子数: {len(test_molecules[0]["positions"])}')

## 2. 安定性解析の詳細

分子の安定性は、各原子の結合数が化学的に妥当かどうかで判定します。

### 2.1 許容結合数の定義

In [ ]:
# 許容結合数の表示
print('\n=== 許容結合数（フォールバックなし） ===\n')

for atom_symbol, allowed in bond_analyze.allowed_bonds.items():
    if isinstance(allowed, int):
        bonds_str = str(allowed)
    else:
        bonds_str = ', '.join(map(str, allowed))
    print(f'{atom_symbol:3s}: {bonds_str} 本の結合')

print('\n注意: これらの値は厳密に適用されます。')
print('      近似的な許容や自動修正は一切行われません。')

### 2.2 個別分子の安定性チェック

In [ ]:
# 個別分子の詳細な安定性チェック
print('\n=== 個別分子の安定性チェック ===\n')

for i, mol in enumerate(test_molecules[:3], 1):  # 最初の3分子のみ表示
    positions = mol['positions']
    atom_types = mol['atom_types']
    
    # 安定性チェック（デバッグモード有効）
    is_stable, nr_stable, total_atoms = check_stability(
        positions, atom_types, dataset_info, debug=True
    )
    
    # 原子タイプの表示
    atom_symbols = [dataset_info['atom_decoder'][at] for at in atom_types]
    
    print(f'分子 {i}:')
    print(f'  原子数: {total_atoms}')
    print(f'  原子タイプ: {", ".join(atom_symbols)}')
    print(f'  安定な原子: {nr_stable}/{total_atoms}')
    print(f'  分子の安定性: {"✓ 安定" if is_stable else "✗ 不安定"}')
    
    if not is_stable:
        print(f'  → 不安定分子は自動修正されません（フォールバックなし）')
    print()

### 2.3 バッチ安定性解析

In [ ]:
# 全分子の安定性解析
print('\n=== バッチ安定性解析 ===\n')

stable_count = 0
atom_stable_count = 0
total_atom_count = 0

for mol in test_molecules:
    is_stable, nr_stable, total_atoms = check_stability(
        mol['positions'], mol['atom_types'], dataset_info, debug=False
    )
    
    if is_stable:
        stable_count += 1
    
    atom_stable_count += nr_stable
    total_atom_count += total_atoms

molecule_stability_rate = stable_count / len(test_molecules) * 100
atom_stability_rate = atom_stable_count / total_atom_count * 100

print(f'分子レベル安定性: {stable_count}/{len(test_molecules)} ({molecule_stability_rate:.1f}%)')
print(f'原子レベル安定性: {atom_stable_count}/{total_atom_count} ({atom_stability_rate:.1f}%)')

print('\n品質ベンチマーク:')
print('  • 優秀: 分子安定性 >95%')
print('  • 良好: 分子安定性 >90%')
print('  • 要改善: 分子安定性 <90%')

## 3. RDKit統合と検証

RDKitを使用して、生成された分子の化学的妥当性を検証します。

### 3.1 RDKit利用可能性の確認

In [ ]:
# RDKitのインポートを試行
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, AllChem
    from qm9.rdkit_functions import build_molecule, mol2smiles
    RDKIT_AVAILABLE = True
    print('✓ RDKit is available')
except ImportError:
    RDKIT_AVAILABLE = False
    print('✗ RDKit not available. Install with: pip install rdkit')
    print('RDKitなしでは一部の評価機能が制限されます。')

### 3.2 分子の妥当性検証

In [ ]:
if RDKIT_AVAILABLE:
    print('\n=== RDKit妥当性検証 ===\n')
    
    valid_molecules = []
    invalid_reasons = []
    
    for i, mol_data in enumerate(test_molecules, 1):
        positions = torch.from_numpy(mol_data['positions']).float()
        atom_types = torch.from_numpy(mol_data['atom_types']).long()
        
        try:
            # RDKit分子オブジェクトを構築（フォールバックなし）
            rdkit_mol = build_molecule(positions, atom_types, dataset_info)
            
            if rdkit_mol is not None:
                # Sanitization（化学的妥当性チェック）
                Chem.SanitizeMol(rdkit_mol)
                
                # SMILES文字列に変換
                smiles = mol2smiles(rdkit_mol)
                
                if smiles and len(smiles.strip()) > 0:
                    valid_molecules.append({
                        'id': i,
                        'mol': rdkit_mol,
                        'smiles': smiles
                    })
                    print(f'✓ 分子 {i}: 有効 (SMILES: {smiles})')
                else:
                    invalid_reasons.append((i, 'Invalid SMILES'))
                    print(f'✗ 分子 {i}: 無効なSMILES')
            else:
                invalid_reasons.append((i, 'Build failed'))
                print(f'✗ 分子 {i}: 分子構築失敗')
                
        except Exception as e:
            invalid_reasons.append((i, str(e)))
            print(f'✗ 分子 {i}: エラー - {str(e)[:50]}')
    
    validity_rate = len(valid_molecules) / len(test_molecules) * 100
    print(f'\n妥当性率: {len(valid_molecules)}/{len(test_molecules)} ({validity_rate:.1f}%)')
    print('\n重要: 無効な分子は自動修正されません（厳密性維持）')
else:
    print('RDKit検証をスキップ（インストールされていません）')

## 4. 一意性（Uniqueness）評価

生成された分子の多様性を評価します。

In [ ]:
if RDKIT_AVAILABLE and len(valid_molecules) > 0:
    print('\n=== 一意性評価 ===\n')
    
    # SMILES文字列のセットを作成
    all_smiles = [mol['smiles'] for mol in valid_molecules]
    unique_smiles = set(all_smiles)
    
    print(f'総分子数: {len(all_smiles)}')
    print(f'一意な分子数: {len(unique_smiles)}')
    
    uniqueness = len(unique_smiles) / len(all_smiles) * 100
    print(f'一意性率: {uniqueness:.1f}%')
    
    # 重複の検出
    smiles_counts = {}
    for smiles in all_smiles:
        smiles_counts[smiles] = smiles_counts.get(smiles, 0) + 1
    
    duplicates = {s: c for s, c in smiles_counts.items() if c > 1}
    
    if duplicates:
        print(f'\n重複検出: {len(duplicates)} 種類')
        for smiles, count in list(duplicates.items())[:3]:  # 最初の3つのみ表示
            print(f'  • {smiles}: {count}回生成')
    else:
        print('\n✓ 重複なし - 完全に一意')
    
    print('\n品質ベンチマーク:')
    print('  • 優秀: 一意性 >98%')
    print('  • 良好: 一意性 >95%')
    print('  • 要改善: 一意性 <95%')
else:
    print('一意性評価をスキップ（有効な分子がありません）')

## 5. プロパティ分布解析

分子のプロパティ（分子量、logP など）の分布を解析します。

In [ ]:
if RDKIT_AVAILABLE and len(valid_molecules) > 0:
    print('\n=== プロパティ分布解析 ===\n')
    
    # 分子プロパティの計算
    properties = {
        'molecular_weight': [],
        'logP': [],
        'num_atoms': [],
        'num_bonds': [],
    }
    
    for mol_info in valid_molecules:
        mol = mol_info['mol']
        
        # 各プロパティを厳密に計算
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        n_atoms = mol.GetNumAtoms()
        n_bonds = mol.GetNumBonds()
        
        properties['molecular_weight'].append(mw)
        properties['logP'].append(logp)
        properties['num_atoms'].append(n_atoms)
        properties['num_bonds'].append(n_bonds)
    
    # 統計情報の表示
    for prop_name, values in properties.items():
        mean_val = np.mean(values)
        std_val = np.std(values)
        min_val = np.min(values)
        max_val = np.max(values)
        
        print(f'{prop_name}:')
        print(f'  平均: {mean_val:.2f} ± {std_val:.2f}')
        print(f'  範囲: [{min_val:.2f}, {max_val:.2f}]')
        print()
else:
    print('プロパティ分布解析をスキップ')

## 6. 可視化

プロパティ分布を可視化します。

In [ ]:
if RDKIT_AVAILABLE and len(valid_molecules) > 0:
    print('\n=== プロパティ分布の可視化 ===\n')
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('生成分子のプロパティ分布', fontsize=14)
    
    # 分子量
    axes[0, 0].hist(properties['molecular_weight'], bins=15, alpha=0.7, edgecolor='black')
    axes[0, 0].set_xlabel('Molecular Weight (g/mol)')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('分子量分布')
    axes[0, 0].grid(True, alpha=0.3)
    
    # logP
    axes[0, 1].hist(properties['logP'], bins=15, alpha=0.7, edgecolor='black', color='green')
    axes[0, 1].set_xlabel('logP')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('親油性分布')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 原子数
    axes[1, 0].hist(properties['num_atoms'], bins=15, alpha=0.7, edgecolor='black', color='orange')
    axes[1, 0].set_xlabel('Number of Atoms')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('原子数分布')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 結合数
    axes[1, 1].hist(properties['num_bonds'], bins=15, alpha=0.7, edgecolor='black', color='red')
    axes[1, 1].set_xlabel('Number of Bonds')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('結合数分布')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('tutorial5_property_distributions.png', dpi=150, bbox_inches='tight')
    print('✓ 可視化を保存しました: tutorial5_property_distributions.png')
    plt.show()
else:
    print('可視化をスキップ')

## 7. 包括的評価スクリプト

実際の使用では、専用スクリプトを使用して大規模評価を行います。

In [ ]:
print('\n=== 包括的評価コマンド ===\n')

eval_command = '''python eval_analyze.py \\\n    --model_path outputs/my_model \\\n    --n_samples 10000 \\\n    --save_molecules True \\\n    --batch_size 100
'''

print('基本的な評価:')
print(eval_command)

print('\n計算されるメトリクス:')
metrics = [
    ('分子レベル安定性', 'Molecule-level stability', '>95%'),
    ('原子レベル安定性', 'Atom-level stability', '>98%'),
    ('妥当性', 'Validity (RDKit)', '>90%'),
    ('一意性', 'Uniqueness', '>98%'),
    ('新規性', 'Novelty', '>95%'),
]

for i, (name_ja, name_en, target) in enumerate(metrics, 1):
    print(f'{i}. {name_ja} ({name_en}): 目標 {target}')

print('\n注意事項:')
print('  • 全メトリクスは厳密に計算されます')
print('  • フォールバック処理は行われません')
print('  • 失敗した計算は明示的にレポートされます')

## まとめ

### 学習した内容

✅ **安定性メトリクスの詳細計算**
   - 許容結合数の厳密な適用
   - 分子レベル・原子レベル評価
   - デバッグモードでの詳細解析

✅ **RDKit統合と検証**
   - 分子オブジェクトの構築
   - Sanitization（化学的妥当性チェック）
   - SMILES変換

✅ **一意性・新規性評価**
   - SMILES重複除去
   - 多様性の定量化
   - ベンチマークとの比較

✅ **プロパティ分布解析**
   - 分子量、logP、原子数、結合数
   - 統計情報の計算
   - 期待値との比較

✅ **可視化手法**
   - ヒストグラム
   - 分布図
   - 品質レポート

✅ **包括的評価パイプライン**
   - 大規模サンプルの評価
   - 複数メトリクスの同時計算
   - 結果の保存と分析

### 重要な原則

1. **厳密性**: 全ての評価は数学的に厳密
2. **透明性**: エラーは明示的に報告
3. **再現性**: 同じ入力は同じ結果
4. **フォールバックなし**: 自動修正は行わない

### 品質ベンチマーク

| メトリクス | 優秀 | 良好 | 要改善 |
|-----------|------|------|--------|
| 分子安定性 | >95% | >90% | <90% |
| 妥当性 | >90% | >85% | <85% |
| 一意性 | >98% | >95% | <95% |
| 新規性 | >95% | >90% | <90% |

### 次のステップ

次のチュートリアルでは、結晶構造の高度な条件付け手法を学びます：
- 複合条件付け戦略
- 空間群制約
- 密度ターゲティング
- 多形生成

**Next: Tutorial 6 - 高度な結晶条件付け**